# Ethnicty mitigation 

In [ ]:
#libraries
import pandas as pd
import numpy as np
import os
import sys
import joblib
import warnings
# from datetime import date, datetime
import pickle
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.linear_model import ElasticNet
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import GroupKFold
from sklearn.model_selection import LeavePGroupsOut
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.utils import resample

import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import pearsonr, spearmanr, norm
import scipy.stats as st
from scipy.stats import zscore as  zscore
from joblib import Parallel, delayed
# from multiprocessing import Manager
from sklearn.cross_decomposition import PLSRegression
import gc
import traceback
import logging
from sklearn.model_selection import KFold
import random

from scipy.stats import gaussian_kde
import matplotlib.ticker as ticker
import matplotlib.colors as mcolors



/home/farzane/anaconda3/envs/nilearn_py11/lib/python3.11/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [ ]:
# Phenotype Labels
l1_labels = {
    'conn_mid': 'MID FC',
    'gfc': 'General FC',
    'tfc': 'Multitask FC',
    'conn_rest': 'Rest FC',
    'conn_wm': 'Nback FC',
    'conn_sst': 'SST FC',
    'T2_gray': 'T2 Gray Matter Avg Intensity',
    'subnet_rest': 'Rest Subcortical-Net FC',
    'cntr_Stop-CorrectGo_task-SST': 'Glasser SST Stop-CorrectGo',
    'T1_gray': 'T1 Gray Matter Avg Intensity',
    'T2_white': 'T2 White Matter Avg Intensity',
    'cntr_IncorrectStop-CorrectGo_task-SST': 'Glasser SST IncorrectStop-CorrectGo',
    'cntr_IncorrectGo_task-SST': 'Glasser SST IncorrectGo',
    'cntr_IncorrectStop_task-SST': 'Glasser SST IncorrectStop',
    'cntr_IncorrectGo-CorrectGo_task-SST': 'Glasser SST IncorrectGo-CorrectGo',
    'T1_norm': 'T1 Normalised Intensity',
    'cntr_CorrectStop_task-SST': 'Glasser SST CorrectStop',
    'T1_white': 'T1 White Matter Avg Intensity',
    'T2_norm': 'T2 Normalised Intensity',
    'cntr_CorrectStop-CorrectGo_task-SST': 'Glasser SST CorrectStop-CorrectGo',
    'surf': 'Surface Area',
    'cntr_IncorrectGo-IncorrectStop_task-SST': 'Glasser SST IncorrectGo-IncorrectStop',
    'cntr_CorrectGo_task-SST': 'Glasser SST CorrectGo',
    'cntr_CorrectStop-IncorrectStop_task-SST': 'Glasser SST CorrectStop-IncorrectStop',
    'T1_summ': 'T1 Summations',
    'VolBrain': 'FreeSurfer Summations',
    'DTI': 'DTI',
    'T2_summ': 'T2 Summations',
    # 'Sulcal_Depth': 'Sulcal Depth',
    'Avg_T1_ASEG_Vol_': 'T1 Subcortical Volume',
    'Dest_Thick_': 'Cortical Thickness',
    'Avg_T2_ASEG_': 'T2 Subcortical Volume',
    'rsmri_within_avg_data': 'rest Cortical-Net FC',
    'Dest_Vol_': 'Cortical Volume',
    'rsmri_gordon_aseg_data': 'rest Temporal Variance',
    'antiLargeRewVsSmallRew_ROI_mid': 'Destriuex MID LargeReward-SmallReward',
    'feedPunPosVsNeg_ROI_mid': 'Destriuex MID LossHit-LossMiss',
    'incorrectgovsincorrectstop_ROI_sst': 'Destriuex SST IncorrectGo-IncorrectStop',
    'anystopvscorrectgo_ROI_sst': 'Destriuex SST AnyStop-CorrectStop',
    'incorrectstopvscorrectgo_ROI_sst': 'Destriuex SST IncorrectStop-CorrectGo',
    'emotionvsneutface_ROI_nbk': 'Destriuex Nback EmotionFace-NeutFace',
    'incorrectgovscorrectgo_ROI_sst': 'Destriuex SST IncorrectGo-CorrectGo',
    'correctgovsfixation_ROI_sst': 'Destriuex SST CorrectGo-Fixation',
    'antiRewVsNeu_ROI_mid': 'Destriuex MID Reward-Neutral',
    'X2back_ROI_nbk': 'Destriuex Nback 2back',
    'facevsplace_ROI_nbk': 'Destriuex Nback Face-Place',
    'antiSmallLossVsNeu_ROI_mid': 'Destriuex MID SmallLoss-Neutral',
    'negfacevsneutface_ROI_nbk': 'Destriuex Nback NegFace-NeutFace',
    'antiLargeLossVsNeu_ROI_mid': 'Destriuex MID LargeLoss-Neutral',
    'emotion_ROI_nbk': 'Destriuex Nback EmotionFace',
    'correctstopvsincorrectstop_ROI_sst': 'Destriuex SST CorrectStop-IncorrectStop',
    'antiSmallRewVsNeu_ROI_mid': 'Destriuex MID SmallReward-Neutral',
    'antiLargeRewVsNeu_ROI_mid': 'Destriuex MID LargeReward-Neutral',
    'antiLosVsNeu_ROI_mid': 'Destriuex MID Loss-Neutral',
    'correctstopvscorrectgo_ROI_sst': 'Destriuex SST CorrectStop-CorrectGo',
    'posfacevsneutface_ROI_nbk': 'Destriuex Nback PosFace-NeutFace',
    'X0back_ROI_nbk': 'Destriuex Nback 0back',
    'place_ROI_nbk': 'Destriuex Nback Place',
    'antiLargeLossVsSmallLoss_ROI_mid': 'Destriuex MID LargeLoss-SmallLoss',
    'X2backvs0back_ROI_nbk': 'Destriuex Nback 2-0back',
    'feedRewPosVsNeg_ROI_mid': 'Destriuex MID RewardHit-RewardMiss',
    'artr_twobk_task-nback': 'Glasser Nback 2back',
    'artr_LossHit-LossMiss_task-MID': 'Glasser MID LossHit-LossMiss',
    'artr_SmallLoss-Neutral_task-MID': 'Glasser MID SmallLoss-Neutral',
    'artr_LgReward-SmallReward_task-MID': 'Glasser MID LargeReward-SmallReward',
    'artr_Loss-Neutral_task-MID': 'Glasser MID Loss-Neutral',
    'artr_PosFace-NeutFace_task-nback': 'Glasser Nback PosFace-NeutFace',
    'artr_LgLoss-Neutral_task-MID': 'Glasser MID LargeLoss-Neutral',
    'artr_NegFace-NeutFace_task-nback': 'Glasser Nback NegFace-NeutFace',
    'artr_face-place_task-nback': 'Glasser Nback Face-Place',
    'artr_LgReward-Neutral_task-MID': 'Glasser MID LargeReward-Neutral',
    'artr_RewardHit-RewardMiss_task-MID': 'Glasser MID RewardHit-RewardMiss',
    'artr_place_task-nback': 'Glasser Nback Place',
    'artr_twobk-zerobk_task-nback': 'Glasser Nback 2-0back',
    'artr_LgLoss-SmallLoss_task-MID': 'Glasser MID LargeLoss-SmallLoss',
    'artr_face_task-nback': 'Glasser Nback Face',
    'artr_emotionface_task-nback': 'Glasser Nback EmotionFace',
    'artr_emotionface-NeutFace_task-nback': 'Glasser Nback Emotionface-NeutFace',
    'artr_zerobk_task-nback': 'Glasser Nback 0back',
    'artr_SmallReward-Neutral_task-MID': 'Glasser MID SmallReward-Neutral',
    'artr_Reward-Neutral_task-MID': 'Glasser MID Reward-Neutral'
}

In [ ]:
# wb class 
from adapt.instance_based import BalancedWeighting
class SafeBalancedWeighting(BalancedWeighting):
    def fit_estimator(self, X, y, **fit_params):
        super().fit_estimator(X, y, **fit_params)
        if not hasattr(self, "sample_weight_") or self.sample_weight_ is None:
            self.sample_weight_ = np.ones(X.shape[0], dtype=float)
        self.sample_weight_ = np.clip(self.sample_weight_, 1e-8, None)
        return self


In [ ]:
# Create MAE/R2 dicts for Adapted and Non-Adapted (used for subsequent analysis and plotting)
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error
from scipy.stats import ttest_rel
import math
import gc
# ----------------------------- PATHS ---------------------------------
root = '/media/hcs-sci-psy-narun/ABCC/fmriresults01/derivatives/ML_Tables/Ethnicity/tsp'
folds = ['Fold_0', 'Fold_1']
modalities = ['abccCntr', 'abccGtfcN', 'abccSmri', 'abcdCntr', 'abcdRsmri', 'abccConnN']  # 

# ----------------------------- DEMO (AA = 2) --------------------------
demo = pd.read_csv(
    '/media/hcs-sci-psy-narun/ABCC/fmriresults01/derivatives/ML_Tables/demo_nesi.csv',
    index_col=0
)
aa_ids = set(demo[demo['race_ethnicity'] == 2].index.astype(str))
print(f"Loaded {len(aa_ids)} AA subjects")

# ----------------------------- FILE PATTERNS --------------------------
typical_pattern = "{fold}/pls/TRB/total_{mod}_pls_output_std_All_noDA_incr30f.joblib"

adapted_patterns = {
    'TRBS2': "{fold}/pls/TRB/total_{mod}_pls_output_std_All_TRBS2_incr10f.joblib",
    'BW':    "{fold}/pls/TRB/total_{mod}_pls_output_std_All_BW_incr10f.joblib",
    'LinInt': "{fold}/pls/TRB/total_{mod}_pls_output_std_All_LinInt_incr10f.joblib",
    'Pred': "{fold}/pls/TRB/total_{mod}_pls_output_std_All_PRED_incr10f.joblib"
}

# Colors and markers for consistent styling
model_styles = {
    'Typical': {'color': 'black',   'marker': 'o', 'ls': '-',  'label': 'Typical PLS'},
    'TRBS2':   {'color': '#1f77b4', 'marker': 'o', 'ls': '-', 'label': 'TRBS2'},
    'BW':      {'color': '#2ca02c', 'marker': 'o', 'ls': '-', 'label': 'BW'},
    'LinInt':  {'color': '#ff7f0e', 'marker': 'o', 'ls': '-',  'label': 'LinInt'},
    'Pred':  {'color': '#9E2A3A', 'marker': 'o', 'ls': '-',  'label': 'Pred'}
}

# =====================================================================
# LOAD RESULTS
# =====================================================================
def load_results_for_modality(mod):
    #results = {'Typical': {}}
    results = {}
    for name, pattern in adapted_patterns.items():
        results[name] = {}

    for fold in folds:
        print(fold)
        # --- 1. Process 'Typical' ---
        t_path = os.path.join(root, typical_pattern.format(fold=fold, mod=mod))
        results['Typical'][fold] = joblib.load(t_path)
        
        # # Access the dictionary 
        current_dict = results['Typical'][fold] 
        for k in current_dict:
            for sk in current_dict[k]:
                data_ref = current_dict[k][sk].get('data', {})
                data_ref.pop('Xtrain', None)
                data_ref.pop('Xtest1', None)
        print('typ loaded')
        # --- 2. Process 'Adapted' patterns ---
        for name, pattern in adapted_patterns.items():
            a_path = os.path.join(root, pattern.format(fold=fold, mod=mod))
            results[name][fold] = joblib.load(a_path)
            print(name, 'adapt loaded')
            # Access the dictionary 
            current_dict = results[name][fold]
            for k in current_dict:
                print(k)
                for sk in current_dict[k]:
                    data_ref = current_dict[k][sk].get('data', {})
                    if 'Xtrain' in data_ref.keys():
                        data_ref.pop('Xtrain', None)
                        data_ref.pop('Xtest1', None)
        
        # Clean up memory after processing all patterns for this fold
        gc.collect()
        
    return results

# =====================================================================
# EXTRACT MAE PER REPETITION (concatenated across folds)
# =====================================================================
def extract_mae_concatenated(results_by_fold):
    tmp = {}
    for fold in folds:
        res = results_by_fold[fold]
        for feat_key, sizes in res.items():
            print(feat_key)
            if feat_key not in tmp:
                tmp[feat_key] = {}
            for size_key, block in sizes.items():
                aa_n = int(size_key.replace("AA_", ""))
                if aa_n not in tmp[feat_key]:
                    tmp[feat_key][aa_n] = {}
                selections = block['data']['selections']
                ytrue_fold = block['data']['yttest']
                idx = ytrue_fold.index.astype(str)
                mask_aa = np.array([i in aa_ids for i in idx])
                ytrue_aa = ytrue_fold.values.flatten()[mask_aa]

                for rep_k, rep_block in selections.items():
                    if rep_k not in tmp[feat_key][aa_n]:
                        tmp[feat_key][aa_n][rep_k] = {'true': {}, 'pred': {}}
                    yp = rep_block['yptest']
                    ypred_aa = yp.values.flatten()[mask_aa]
                    tmp[feat_key][aa_n][rep_k]['true'][fold] = ytrue_aa
                    tmp[feat_key][aa_n][rep_k]['pred'][fold] = ypred_aa

    # Concatenate and compute MAE per repetition
    out = {}
    for feat_key, sizes in tmp.items():
        out[feat_key] = {}
        for aa_n, reps in sizes.items():
            rep_maes = []
            for rep_k, rep_block in reps.items():
                ytrue = np.concatenate([rep_block['true'][f] for f in folds])
                ypred = np.concatenate([rep_block['pred'][f] for f in folds])
                if model_name in ['LinInt', 'Pred'] and aa_n in [10, 20]:
                    rep_maes.append(np.array([]))  # empty → will skip point
                else:
                    rep_maes.append(mean_absolute_error(ytrue, ypred))
            out[feat_key][aa_n] = rep_maes
            gc.collect()
    return out


# =====================================================================
# MAIN 
# =====================================================================
out_dir = os.path.join(root, 'plots')
os.makedirs(out_dir, exist_ok=True)

for mod in modalities:
    all_results = load_results_for_modality(mod)
    print(mod)
    # Extract MAEs for all models
    mae_data = {}
    for model_name, results_by_fold in all_results.items():
        print(model_name)
        mae_data[model_name] = extract_mae_concatenated(results_by_fold)
        joblib.dump(mae_data, root + f'{mod}_MAE-ADAPTYP.joblib')
        gc.collect()


Loaded 1768 AA subjects
Fold_0
FA adapt loaded
Pred adapt loaded
Fold_1
FA adapt loaded
Pred adapt loaded
abccCntr
FA
cntr_Stop-CorrectGo_task-SST
cntr_CorrectStop-IncorrectStop_task-SST
cntr_CorrectGo_task-SST
cntr_CorrectStop-CorrectGo_task-SST
cntr_IncorrectStop-CorrectGo_task-SST
cntr_IncorrectGo_task-SST
cntr_IncorrectStop_task-SST
cntr_IncorrectGo-IncorrectStop_task-SST
cntr_CorrectStop_task-SST
cntr_IncorrectGo-CorrectGo_task-SST
artr_twobk_task-nback
artr_LossHit-LossMiss_task-MID
artr_SmallLoss-Neutral_task-MID
artr_LgReward-SmallReward_task-MID
artr_Loss-Neutral_task-MID
artr_PosFace-NeutFace_task-nback
artr_LgLoss-Neutral_task-MID
artr_NegFace-NeutFace_task-nback
artr_face-place_task-nback
artr_LgReward-Neutral_task-MID
artr_RewardHit-RewardMiss_task-MID
artr_place_task-nback
artr_twobk-zerobk_task-nback
artr_LgLoss-SmallLoss_task-MID
artr_face_task-nback
artr_emotionface_task-nback
artr_emotionface-NeutFace_task-nback
artr_zerobk_task-nback
artr_SmallReward-Neutral_task-MID

In [ ]:
# gap calculations
import os
import gc
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ----------------------------- PATHS ---------------------------------
root = '/media/hcs-sci-psy-narun/ABCC/fmriresults01/derivatives/ML_Tables/Ethnicity/tsp'
folds = ['Fold_0', 'Fold_1']
modalities = ['abccCntr', 'abccGtfcN', 'abccSmri', 'abcdCntr', 'abcdRsmri', 'abccConnN']

out_dir = os.path.join(root, 'plots')
os.makedirs(out_dir, exist_ok=True)

# ----------------------------- DEMO ----------------------------------
demo = pd.read_csv(
    '/media/hcs-sci-psy-narun/ABCC/fmriresults01/derivatives/ML_Tables/demo_nesi.csv',
    index_col=0
)

aa_ids = set(demo[demo['race_ethnicity'] == 2].index.astype(str))
wa_ids = set(demo[demo['race_ethnicity'] == 1].index.astype(str))

print(f"Loaded {len(aa_ids)} AA subjects")
print(f"Loaded {len(wa_ids)} WA subjects")

# ----------------------------- FILE PATTERN --------------------------
typical_pattern = "{fold}/pls/TRB/total_{mod}_pls_output_std_All_noDA_incr30f.joblib"


# =====================================================================
# HELPERS
# =====================================================================
def load_typical_results(mod):
    out = {}
    for fold in folds:
        path = os.path.join(root, typical_pattern.format(fold=fold, mod=mod))
        print(f"Loading Typical PLS: {path}")
        out[fold] = joblib.load(path)
        gc.collect()
    return out


def subject_bootstrap_gap(ytrue_aa, ypred_aa, ytrue_wa, ypred_wa,
                          n_boot=5000, ci=95, seed=0):
    """
    Bootstrap over subjects within AA and WA separately.
    Gap = (MAE_AA - MAE_WA) / MAE_AA  
    """
    ytrue_aa = np.asarray(ytrue_aa).flatten()
    ypred_aa = np.asarray(ypred_aa).flatten()
    ytrue_wa = np.asarray(ytrue_wa).flatten()
    ypred_wa = np.asarray(ypred_wa).flatten()

    abs_err_aa = np.abs(ytrue_aa - ypred_aa)
    abs_err_wa = np.abs(ytrue_wa - ypred_wa)

    mae_aa = abs_err_aa.mean()
    mae_wa = abs_err_wa.mean()

    gap = (mae_aa - mae_wa) / mae_aa if mae_aa > 0 else np.nan

    # bootstrap
    rng = np.random.default_rng(seed)
    boots = []

    for _ in range(n_boot):
        samp_aa = rng.choice(abs_err_aa, size=len(abs_err_aa), replace=True)
        samp_wa = rng.choice(abs_err_wa, size=len(abs_err_wa), replace=True)

        b_mae_aa = samp_aa.mean()
        b_mae_wa = samp_wa.mean()

        if b_mae_aa <= 0:
            continue

        boots.append((b_mae_aa - b_mae_wa) / b_mae_aa)

    boots = np.asarray(boots, dtype=float)
    if len(boots) == 0:
        return gap, (np.nan, np.nan), mae_aa, mae_wa

    alpha = (100 - ci) / 2
    lo = np.percentile(boots, alpha)
    hi = np.percentile(boots, 100 - alpha)

    return gap, (lo, hi), mae_aa, mae_wa


def extract_step0_rep0_concat(typical_results_by_fold, phenotype, step0_key="AA_0", rep_key=None):
    """
    Returns concatenated ytrue/ypred for AA and WA across folds for ONE repetition.
    """

    ytrue_aa_all, ypred_aa_all = [], []
    ytrue_wa_all, ypred_wa_all = [], []

    for fold in folds:
        res = typical_results_by_fold[fold]
        if phenotype not in res:
            continue
        if step0_key not in res[phenotype]:
            continue

        block = res[phenotype][step0_key]
        selections = block["data"]["selections"]

        # choose rep 0 automatically if not given
        if rep_key is None:
            rep_key = sorted(selections.keys())[0]

        ytrue_fold = block["data"]["yttest"]
        idx = ytrue_fold.index.astype(str)

        mask_aa = np.array([i in aa_ids for i in idx])
        mask_wa = np.array([i in wa_ids for i in idx])

        ytrue = ytrue_fold.values.flatten()

        yp = selections[rep_key]["yptest"].values.flatten()

        ytrue_aa_all.append(ytrue[mask_aa])
        ypred_aa_all.append(yp[mask_aa])

        ytrue_wa_all.append(ytrue[mask_wa])
        ypred_wa_all.append(yp[mask_wa])

    if len(ytrue_aa_all) == 0 or len(ytrue_wa_all) == 0:
        return None

    return (
        np.concatenate(ytrue_aa_all), np.concatenate(ypred_aa_all),
        np.concatenate(ytrue_wa_all), np.concatenate(ypred_wa_all)
    )



# =====================================================================
# RUN (ALL MODALITIES + ALL PHENOTYPES)
# =====================================================================
rows = []

for mod in modalities:
    typical_results_by_fold = load_typical_results(mod)

    # collect phenotype list from fold 0
    phenotypes = sorted(typical_results_by_fold[folds[0]].keys())

    for phenotype in phenotypes:
        extracted = extract_step0_rep0_concat(
            typical_results_by_fold,
            phenotype=phenotype,
            step0_key="AA_0",
            rep_key=None  # auto pick first rep (rep0)
        )

        if extracted is None:
            continue

        ytrue_aa, ypred_aa, ytrue_wa, ypred_wa = extracted

        gap, (lo, hi), mae_aa, mae_wa = subject_bootstrap_gap(
            ytrue_aa, ypred_aa, ytrue_wa, ypred_wa,
            n_boot=5000, seed=0
        )

        rows.append({
            "modality": mod,
            "phenotype": phenotype,
            "n_AA": len(ytrue_aa),
            "n_WA": len(ytrue_wa),
            "MAE_AA": mae_aa,
            "MAE_WA": mae_wa,
            "gap": gap,
            "gap_ci_low": lo,
            "gap_ci_high": hi
        })

    gc.collect()

df_all = pd.DataFrame(rows)

# rank globally
df_all = df_all.sort_values("gap", ascending=False).reset_index(drop=True)
df_all["global_rank"] = np.arange(1, len(df_all) + 1)

# save combined table
table_path = os.path.join(out_dir, "ALLMODS_TypicalPLS_Step0_GapRanking_rep0_SUBJECTBOOT.csv")
df_all.to_csv(table_path, index=False)
print(f"Saved table: {table_path}")



Loaded 1768 AA subjects
Loaded 6163 WA subjects
Loading Typical PLS: /media/hcs-sci-psy-narun/ABCC/fmriresults01/derivatives/ML_Tables/Ethnicity/tsp/Fold_0/pls/TRB/total_abccCntr_pls_output_std_All_noDA_incr30f.joblib
Loading Typical PLS: /media/hcs-sci-psy-narun/ABCC/fmriresults01/derivatives/ML_Tables/Ethnicity/tsp/Fold_1/pls/TRB/total_abccCntr_pls_output_std_All_noDA_incr30f.joblib
Loading Typical PLS: /media/hcs-sci-psy-narun/ABCC/fmriresults01/derivatives/ML_Tables/Ethnicity/tsp/Fold_0/pls/TRB/total_abccGtfcN_pls_output_std_All_noDA_incr30f.joblib
Loading Typical PLS: /media/hcs-sci-psy-narun/ABCC/fmriresults01/derivatives/ML_Tables/Ethnicity/tsp/Fold_1/pls/TRB/total_abccGtfcN_pls_output_std_All_noDA_incr30f.joblib
Loading Typical PLS: /media/hcs-sci-psy-narun/ABCC/fmriresults01/derivatives/ML_Tables/Ethnicity/tsp/Fold_0/pls/TRB/total_abccSmri_pls_output_std_All_noDA_incr30f.joblib
Loading Typical PLS: /media/hcs-sci-psy-narun/ABCC/fmriresults01/derivatives/ML_Tables/Ethnicity/tsp

In [ ]:
# load gap table
df_all = pd.read_csv('/media/hcs-sci-psy-narun/ABCC/fmriresults01/derivatives/ML_Tables/Ethnicity/tsp/plots/ALLMODS_TypicalPLS_Step0_GapRanking_rep0_SUBJECTBOOT.csv')


In [ ]:
# Gap Ranking
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch


# -------------------------------
# Group assignment logic
# -------------------------------
def assign_group(feature: str) -> str:
    """
    Assign feature group based on phenotype prefix (group by modality)
    """
    if isinstance(feature, str):
        f = feature.lower()

        # Functional connectivity
        if f.startswith(("conn_", "gfc", "tfc", "rsmri", "rest", "subnet")):
            return "FCs"

        # Glasser task contrasts (ABCC)
        elif f.startswith(("cntr_", "artr_")):
            return "Task Contrasts (ABCC)"

        # Destrieux / ABCD-style task contrasts
        elif f.startswith((
            "anti", "feed", "incorrect", "correct",
            "emotion", "posface", "negface",
            "place", "x", "any", "face"
        )):
            return "Task Contrasts (ABCD)"

    # Default fallback
    return "sMRI & DTI"


# -------------------------------
# Color mapping
# -------------------------------
group_colors = {
    "FCs": "tab:blue",
    "Task Contrasts (ABCC)": "tab:orange",
    "Task Contrasts (ABCD)": "tab:green",
    "sMRI & DTI": "tab:red"
}


# -------------------------------
# Plotting function
# -------------------------------
def plot_gap_ranking_combined(df, title, save_path, l1_labels):
    df = df.copy()

    # --- Assign groups and colors ---
    df["group"] = df["phenotype"].apply(assign_group)
    df["color"] = df["group"].map(group_colors)

    df = df[df['phenotype'].isin(l1_labels.keys())]
    # --- Map phenotype to readable labels ---
    df["label"] = df["phenotype"].map(l1_labels).fillna(df["phenotype"])

    # --- warn about missing labels ---
    missing = df.loc[~df["phenotype"].isin(l1_labels.keys()), "phenotype"].unique()
    if len(missing) > 0:
        print("⚠️ Missing labels for:", missing)

    # --- Sort by gap ---
    df = df.sort_values("gap", ascending=True).reset_index(drop=True)

    y = np.arange(len(df))
    x = df["gap"].values

    # --- CI computation ---
    xerr_low = x - df["gap_ci_low"].values
    xerr_high = df["gap_ci_high"].values - x
    xerr = np.vstack([xerr_low, xerr_high])

    # --- Plot ---
    plt.figure(figsize=(14, max(7, 0.35 * len(df))))

    plt.barh(
        y,
        x,
        xerr=xerr,
        capsize=3,
        color=df["color"].values,
        edgecolor="black",
        linewidth=0.5
    )
    # (-1.1, len(df_single) - 0.0)
    plt.ylim((-1.1, len(df)))
    plt.yticks(y, df["label"].values, fontsize=24)
    plt.xticks(fontsize=26)

    plt.axvline(0, linestyle="--", linewidth=1)

    plt.xlabel("")
    plt.title(title)

    # # --- Legend ---
    # legend_elements = [
    #     Patch(facecolor=color, label=group)
    #     for group, color in group_colors.items()
    # ]

    # plt.legend(
    #     handles=legend_elements,
    #     title="Feature Group",
    #     bbox_to_anchor=(1.02, 1),
    #     loc="upper left", fontsize=24
    # )

    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close()

    print(f"Saved: {save_path}")

root = '/media/hcs-sci-psy-narun/ABCC/fmriresults01/derivatives/ML_Tables/Ethnicity/tsp'
table_path = os.path.join(out_dir, "ALLMODS_TypicalPLS_Step0_GapRanking_rep0_SUBJECTBOOT.csv")
df_all = pd.read_csv(table_path)

combined_plot_path = os.path.join(out_dir, "ALLMODS_TypicalPLS_Step0_GapRanking.svg")

plot_gap_ranking_combined(
    df_all,
    title="Typical PLS (Step 0 WA-only) Gap Ranking — All Modalities (bootstrap 95% CI)",
    save_path=combined_plot_path,
    l1_labels=l1_labels
)

In [ ]:
# AA MAE PLOT Line selected
import os
import joblib
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import ttest_rel
import math
import gc
import seaborn as sns

# =====================================================================
# SETTINGS
# =====================================================================
root = '/media/hcs-sci-psy-narun/ABCC/fmriresults01/derivatives/ML_Tables/Ethnicity/'
out_dir = os.path.join(root, 'plots')
os.makedirs(out_dir, exist_ok=True)

#  modality chunks with saved MAE files
modalities = ['abccCntr', 'abccGtfcN', 'abccSmri', 'abcdCntr', 'abcdRsmri', 'abccConnN']

df_all = pd.read_csv('/media/hcs-sci-psy-narun/ABCC/fmriresults01/derivatives/ML_Tables/Ethnicity/tsp/plots/ALLMODS_TypicalPLS_Step0_GapRanking_rep0_SUBJECTBOOT.csv', index_col=1)

selected_features = list(l1_labels.keys())
df_all = df_all.loc[selected_features]
df_all =  df_all.sort_values(by='gap')
least_biased = list(df_all.index)[0:10]
# least_biased.remove('rest')
most_biased = list(df_all.index)[-10:]
selected_features = least_biased + most_biased

print(selected_features)

# Model styles 
model_styles = {
    'Typical': {'color': 'black',     'ls': '-',    'marker': 'o', 'label': 'Standard'},
    'TRBS2':   {'color': '#1f77b4',   'ls': '-',    'marker': 'o', 'label': 'TRBS2'},
    'BW':      {'color': '#2ca02c',   'ls': '-',    'marker': 'o', 'label': 'BW'},
    'LinInt':  {'color': '#ff7f0e',   'ls': '-',    'marker': 'o', 'label': 'LinInt'},
    'Pred':  {'color': '#9E2A3A', 'marker': 'o', 'ls': '-',  'label': 'Pred'}
}

def p_to_star(p):
    if p <= 0.001:
        return '***'
    elif p <= 0.01:
        return '**'
    elif p <= 0.05:
        return '*'
    return ''

# boot function
def bootstrap_ci(vals, n_boot=2000, ci=95, random_state=42):
    if len(vals) == 0:
        return np.nan, np.nan
    
    rng = np.random.default_rng(random_state)
    boot_means = []
    
    for _ in range(n_boot):
        sample = rng.choice(vals, size=len(vals), replace=True)
        boot_means.append(np.mean(sample))
    
    alpha = (100 - ci) / 2
    lower = np.percentile(boot_means, alpha)
    upper = np.percentile(boot_means, 100 - alpha)
    
    return lower, upper
# =====================================================================
# LOAD MAE DATA FROM ALL SAVED JOBLIB FILES
# =====================================================================
mae_data_global = {model: {} for model in model_styles}

for mod in modalities:
    file_path1 = os.path.join(root, f'tsp{mod}_MAE-ADAPTYP.joblib')
    file_path2 = os.path.join(root, f'tsp{mod}_MAE-FADAPTYP.joblib')
    if not os.path.exists(file_path1):
        print(f"Warning: Missing MAE file for {mod}: {file_path1}")
        continue
    
    mae_data_mod1 = joblib.load(file_path1)  # {model: {feat: {aa_n: [mae_values]}}}
    mae_data_mod2 = joblib.load(file_path2)
    mae_data_mod = {**mae_data_mod2, **mae_data_mod1}
    print(mae_data_mod.keys())
    for model_name in model_styles:
        
        if model_name not in mae_data_mod:
            print(model_name, 'not in mae data')
            continue
        for feat_key in selected_features:
            if feat_key in mae_data_mod[model_name]:
                print(feat_key)
                if feat_key not in mae_data_global[model_name]:
                    mae_data_global[model_name][feat_key] = {}
                for aa_n, mae_vals in mae_data_mod[model_name][feat_key].items():
                    if aa_n not in mae_data_global[model_name][feat_key]:
                        mae_data_global[model_name][feat_key][aa_n] = []
                    mae_data_global[model_name][feat_key][aa_n].extend(mae_vals)

print(f"Loaded MAE data for {len(mae_data_global['Typical'])} selected features across modalities.")

# =====================================================================
# SINGLE COMBINED PLOT FOR ALL 20 SELECTED FEATURES
# =====================================================================
sns.set(style="whitegrid", context="talk")

# n_feats = len(selected_features)
# selected_features = list(l1_labels_2.keys())
n_feats = len(selected_features)

n_cols = 4
n_rows = math.ceil(n_feats / n_cols)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(10 * n_cols, 7 * n_rows), sharex=True, sharey=True)
axes = np.array(axes).flatten()

legend_added = False

for i, feat_key in enumerate(selected_features):
    ax = axes[i]
    
    # Check if any model has data for this feature
    has_data = any(feat_key in mae_data_global.get(model, {}) for model in model_styles)
    if not has_data:
        ax.text(0.5, 0.5, "No data", ha='center', va='center', transform=ax.transAxes, fontsize=12)
        ax.set_title(l1_labels.get(feat_key, feat_key), fontsize=22)
        continue
    
    # Collect all AA sample sizes
    all_sizes = set()
    for model_dict in mae_data_global.values():
        if feat_key in model_dict:
            all_sizes.update(model_dict[feat_key].keys())
    aa_sizes = sorted(all_sizes)
    
    # Prepare values per model and size
    plot_data = {}
    for model_name in model_styles:
        vals_per_size = []
        for aa_n in aa_sizes:
            vals = np.array(mae_data_global[model_name].get(feat_key, {}).get(aa_n, []))
            # Skip points for LinInt and Pred at 10 and 20
            if model_name == ['LinInt', 'Pred'] and aa_n in [10, 20]:
                vals_per_size.append(np.array([]))
            else:
                vals_per_size.append(vals)
        plot_data[model_name] = vals_per_size
    
    # Plot lines and confidence intervals
    for model_name, style in model_styles.items():
        means, lowers, uppers, valid_sizes = [], [], [], []
        for aa_n, vals in zip(aa_sizes, plot_data[model_name]):
            if len(vals) == 0:
                continue
            # mean = np.mean(vals)
            # ci = 1.96 * np.std(vals, ddof=1) / np.sqrt(len(vals)) if len(vals) > 1 else 0
            # ci = max(ci, 1e-4)
            # means.append(mean)
            # lowers.append(mean - ci)
            # uppers.append(mean + ci)
            # 
            # mean = np.mean(vals)
            # if len(vals) > 1:
            #     lower, upper = bootstrap_ci(vals, n_boot=2000)
            # else:
            #     lower, upper = mean, mean

            # means.append(mean)
            # lowers.append(lower)
            # uppers.append(upper)
            # valid_sizes.append(aa_n)
            from scipy.stats import bootstrap

            res = bootstrap(
                (vals,),
                np.mean,
                confidence_level=0.95,
                n_resamples=5000,
                method='BCa',
                random_state=42
            )

            lower = res.confidence_interval.low
            upper = res.confidence_interval.high
        
        if len(valid_sizes) == 0:
            continue
        
        ax.plot(valid_sizes, means, marker=style['marker'], color=style['color'],
                ls=style['ls'], label=style['label'], linewidth=4)
        ax.fill_between(valid_sizes, lowers, uppers, color=style['color'], alpha=0.3)
    
    # # Significance stars vs Typical
    # if 'Typical' in plot_data:
    #     typ_vals_list = plot_data['Typical']
    #     for model_name in ['TRBS2', 'BW', 'LinInt']:
    #         if model_name not in plot_data:
    #             continue
    #         color = model_styles[model_name]['color']
    #         adp_vals_list = plot_data[model_name]
    #         for aa_n, typ_vals, adp_vals in zip(aa_sizes, typ_vals_list, adp_vals_list):
    #             if len(typ_vals) < 2 or len(adp_vals) == 0 or len(typ_vals) != len(adp_vals):
    #                 continue
    #             p = ttest_rel(typ_vals, adp_vals).pvalue
    #             star = p_to_star(p)
    #             if star:
    #                 typ_mean = np.mean(typ_vals)
    #                 adp_mean = np.mean(adp_vals)
    #                 y_pos = min(typ_mean, adp_mean) * 0.98
    #                 offset = 0.01 * (ax.get_ylim()[1] - ax.get_ylim()[0])
    #                 ax.text(aa_n, y_pos + offset, star, ha='center', va='top',
    #                         color=color, fontsize=12, fontweight='bold')
    
    # Labels
    ax.set_title(l1_labels.get(feat_key, feat_key), fontsize=48, pad=4 )#, fontsize=24
    if i in [0, 5, 10, 15]:
        ax.set_ylabel("")
    else:
        ax.set_ylabel("")

    # ax.set_ylabel("")
    ax.grid(True, alpha=0.3)
    ax.set_xlabel("")

[ax.set_xlabel("") for ax in axes[-5:]]
# Legend only once
# if not legend_added:
#     plt.legend(title = 'Model' , title_fontsize = 'small' , alignment= 'left',
#         frameon=True, ncol=1, fontsize=16, bbox_to_anchor=(1.02, 1), loc='upper left')
#     legend_added = True
# Remove empty subplots
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])
# axes[0].set_xticks(aa_sizes)
for ax in axes:
    ax.tick_params(axis='y', labelsize=36)
    ax.tick_params(axis='x', labelsize=36)

for ax in axes:
    for spine in ax.spines.values():
        spine.set_edgecolor("black")
        spine.set_linewidth(2)
# plt.tight_layout(rect=[0, 0.05, 0.98, 0.95])
fig.subplots_adjust(
    left=0.05,
    right=0.92,
    top=0.96,
    bottom=0.05,
    hspace=0.38,   
    wspace=0.25    
)
# Save 

save_path_svg = os.path.join(out_dir, "All_Adapted_vs_Typical_AA_MAE_combined_Gap-selected_boot.svg")
fig.savefig(save_path_svg, format='svg', dpi=600)
plt.close(fig)

gc.collect()

In [ ]:
# MAE and R2 Heatmap
import os
import joblib
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib import gridspec
from matplotlib.colors import ListedColormap

# ------------------ Paths & Settings ------------------

root = "/media/hcs-sci-psy-narun/ABCC/fmriresults01/derivatives/ML_Tables/Ethnicity/"

rank_table_dir = (
    "/media/hcs-sci-psy-narun/ABCC/fmriresults01/derivatives/"
    "ML_Tables/Ethnicity/tsp/results_concat_transformed/"
    "tables/ranking_single_stacked.csv"
)

modalities = [
    "tspabccGtfcN",
    "tspabccCntr",
    "tspabccSmri",
    "tspabcdRsmri",
    "tspabcdCntr",
    "tspabccConnN",
]

BASELINE = "Typical"
LININT_MIN_N = 30

# ------------------ Load Bias Ranking ------------------

df_all = pd.read_csv('/media/hcs-sci-psy-narun/ABCC/fmriresults01/derivatives/ML_Tables/Ethnicity/tsp/plots/ALLMODS_TypicalPLS_Step0_GapRanking_rep0_SUBJECTBOOT.csv', index_col=1)
bias_order = (
    df_all["gap"]
    .sort_values(ascending=False)  # Most biased first
    .index.tolist()
)


# ------------------ Restrict to l1_labels ------------------

valid_features = list(l1_labels.keys())

bias_order = [f for f in bias_order if f in valid_features]

# Top / bottom 10
most_bias = set(bias_order[:10])
least_bias = set(bias_order[-10:])

# ------------------ Build MAE Long Table ------------------

rows = []

for mod in modalities:
    # Choose metric to load:
    file_path1 = os.path.join(root, f"{mod}_MAE-ADAPTYP.joblib")
    # file_path1 = os.path.join(root, f"{mod}_R2-ADAPTYP.joblib")
    if not os.path.exists(file_path1):
        print(f"Warning: Missing MAE file for {mod}")
        continue

    mae_data = joblib.load(file_path1)


    for model_name, feats in mae_data.items():
        if model_name == "FA":
            continue
        elif model_name == 'TRBS2':
            model_name = 'TRB'
        for feat_key, sizes in feats.items():
            if feat_key not in valid_features:
                continue

            for aa_n, rep_vals in sizes.items():

                rep_vals = np.asarray(rep_vals, dtype=float)
                rep_vals = rep_vals[~np.isnan(rep_vals)]

                if len(rep_vals) == 0:
                    continue

                rows.append({
                    "phenotype": feat_key,
                    "modality": mod,
                    "adapt_method": model_name,
                    "n_AA": aa_n,
                    "mae_mean": rep_vals.mean()
                })

mae_long = pd.DataFrame(rows)

# ------------------ AUC Benefit ------------------

benefit_auc = []

for ph in mae_long["phenotype"].unique():

    base = mae_long[
        (mae_long["phenotype"] == ph) &
        (mae_long["adapt_method"] == BASELINE)
    ].sort_values("n_AA")

    if len(base) < 2:
        continue

    for method in mae_long["adapt_method"].unique():
        if method == BASELINE:
            continue

        adp = mae_long[
            (mae_long["phenotype"] == ph) &
            (mae_long["adapt_method"] == method)
        ].sort_values("n_AA")

        if method in ["LinInt", "Pred"]:
            adp = adp[adp["n_AA"] >= LININT_MIN_N]
            base_sub = base[base["n_AA"] >= LININT_MIN_N]
        else:
            base_sub = base

        merged = pd.merge(
            base_sub[["n_AA", "mae_mean"]],
            adp[["n_AA", "mae_mean"]],
            on="n_AA",
            suffixes=("_base", "_adapt")
        )

        if len(merged) < 2:
            continue

        improvement = merged["mae_mean_base"] - merged["mae_mean_adapt"]
        auc = np.trapz(improvement, merged["n_AA"])

        benefit_auc.append({
            "phenotype": ph,
            "adapt_method": method,
            "benefit_auc": -auc
        })

benefit_auc_df = pd.DataFrame(benefit_auc)

# ------------------ Pivot & Sort ------------------

pivot_auc = benefit_auc_df.pivot(
    index="phenotype",
    columns="adapt_method",
    values="benefit_auc"
)

# Keep only valid + ranked phenotypes
pivot_auc = pivot_auc.loc[
    [p for p in bias_order if p in pivot_auc.index]
]

# Apply readable labels
pivot_auc.index = pivot_auc.index.map(lambda x: l1_labels.get(x, x))

# ------------------ Side-by-Side Heatmaps, True Values, Single Colorbar ------------------

half = 40
first_half = pivot_auc.iloc[:half, :]
second_half = pivot_auc.iloc[half:half*2, :]

fig, axes = plt.subplots(
    1, 2,
    figsize=(22, 24),
    gridspec_kw={"wspace": 1.9}  # add space between subplots
)

# Compute global min/max for consistent colorbar
vmin = pivot_auc.min().min()*-1
vmax = pivot_auc.max().max()*-1

# ---- First 40 Phenotypes ----
sns.heatmap(
    first_half,
    ax=axes[0],
    cmap="RdBu_r",
    center=0,
    linewidths=0.4,
    cbar=False,
    vmin=vmin,
    vmax=vmax
)
axes[0].set_title("", fontsize=20)
axes[0].set_xlabel("", fontsize=16)
axes[0].set_ylabel("Phenotype", fontsize=16)
axes[0].tick_params(axis='x', labelsize=28)
axes[0].tick_params(axis='y', labelsize=28)

# ---- Second 40 Phenotypes ----
sns.heatmap(
    second_half,
    ax=axes[1],
    cmap="RdBu_r",
    center=0,
    linewidths=0.4,
    cbar=False,
    vmin=vmin,
    vmax=vmax
)
axes[1].set_title("", fontsize=20)
axes[1].set_xlabel("", fontsize=16)
axes[1].set_ylabel("")  # remove y-label to avoid redundancy
axes[1].tick_params(axis='x', labelsize=28)
axes[1].tick_params(axis='y', labelsize=28)

# ---- Shared Colorbar ----
cbar_ax = fig.add_axes([0.92, 0.15, 0.015, 0.7])
cbar = fig.colorbar(
    plt.cm.ScalarMappable(cmap="RdBu_r", norm=plt.Normalize(vmin=vmin, vmax=vmax)),
    cax=cbar_ax
)
cbar.set_label("AUC Benefit (True Values)", fontsize=16)
cbar.ax.tick_params(labelsize=30)

plt.tight_layout(rect=[0, 0, 0.9, 1])
plt.savefig(os.path.join(root, "Phenotypes_FirstLast40_sideBySide_heatmap_MAEtrueValuesGap.svg"), bbox_inches="tight")
plt.close()



In [ ]:
# compare adapt methods
from scipy.stats import wilcoxon
from statsmodels.stats.multitest import multipletests
from itertools import combinations
import pandas as pd
import numpy as np

# Pivot table
pivot = benefit_auc_df.pivot(
    index='phenotype',
    columns='adapt_method',
    values='benefit_auc'
)

methods = pivot.columns

results = []

for m1, m2 in combinations(methods, 2):

    x = pivot[m1]
    y = pivot[m2]

    # Wilcoxon
    stat, p = wilcoxon(
        x,
        y,
        alternative='two-sided'
    )

    # ---- paired Cohen's d ----
    diff = x - y
    cohens_d = diff.mean() / diff.std(ddof=1)

    # ---- rank-biserial correlation ----
    n = len(diff)
    rbc = 1 - (2 * stat) / (n * (n + 1))

    results.append({
        'method_1': m1,
        'method_2': m2,
        'statistic': stat,
        'p_uncorrected': p,
        'cohens_d_paired': cohens_d,
        'rank_biserial_r': rbc
    })

# Results dataframe
results_df = pd.DataFrame(results)

# Multiple comparison correction
reject, p_corrected, _, _ = multipletests(
    results_df['p_uncorrected'],
    method='holm'
)

results_df['p_corrected'] = p_corrected
results_df['significant'] = reject

# stars
def p_to_star(p):
    if p <= 0.001:
        return '***'
    elif p <= 0.01:
        return '**'
    elif p <= 0.05:
        return '*'
    return ''

results_df['stars'] = results_df['p_corrected'].apply(p_to_star)

# Sort
results_df = results_df.sort_values('p_corrected')

print(results_df)

  method_1 method_2  statistic  p_uncorrected  cohens_d_paired  \
0       BW   LinInt       12.0   1.234446e-14         1.823622   
1       BW     Pred       21.0   1.729871e-14         1.960531   
4   LinInt    TRBS2      287.0   1.621832e-10        -0.980475   
5     Pred    TRBS2      399.0   4.733879e-09        -0.743298   
2       BW    TRBS2      422.0   9.140521e-09         0.814881   
3   LinInt     Pred     1146.0   2.299967e-02        -0.046905   

   rank_biserial_r   p_corrected  significant stars  
0         0.996296  7.406675e-14         True   ***  
1         0.993519  8.649356e-14         True   ***  
4         0.911420  6.487327e-10         True   ***  
5         0.876852  1.420164e-08         True   ***  
2         0.869753  1.828104e-08         True   ***  
3         0.646296  2.299967e-02         True     *  


In [ ]:
# plot gain-gap scatter
from scipy.stats import pearsonr
import matplotlib.pyplot as plt
import seaborn as sns

# ------------------ Categorize Phenotypes ------------------
def categorize_pheno(label):
    label = label.lower()
    if 'fc' in label or 'rest' in label:
        return 'Connectivity'
    elif 'destriuex' in label:
        return 'Destrieux Contrasts'
    elif 'glasser' in label:
        return 'Glasser Contrasts'
    else:
        return 'Structural'
# Compute average gain across methods
avg_gain = benefit_auc_df.groupby("phenotype")["benefit_auc"].mean()
# Extract gap score
bias_score = df_all.loc[avg_gain.index, "gap"]
# Ensure alignment
avg_gain = avg_gain.loc[bias_score.index]

# Create category mapping
categories = [categorize_pheno(l1_labels[ph]) for ph in avg_gain.index]

# ------------------ Compute Correlation ------------------
r, p = pearsonr(bias_score, avg_gain)
r2 = r**2

# ------------------ Scatter Plot ------------------
plt.figure(figsize=(10, 8))

# Use seaborn for automatic coloring by category
scatter_df = pd.DataFrame({
    "Bias": bias_score,
    "AvgGain": avg_gain,
    "Category": categories
})

palette = {
    "Connectivity": "dodgerblue",
    "Destrieux Contrasts": "green",
    "Glasser Contrasts": "orange",
    "Structural": "red"
}
sns.set_style("white")
sns.scatterplot(
    data=scatter_df,
    x="Bias",
    y="AvgGain",
    hue="Category",
    palette=palette,
    s=140,
    edgecolor='k',
    alpha=0.8, 
    
)

# Linear fit line (overall, ignoring categories)
m, b = np.polyfit(bias_score, avg_gain, 1)
plt.plot(bias_score, m*bias_score + b, color="black", linestyle="--", linewidth=2, )

# Labels and title
plt.xlabel("Bias Score (abs MAE)", fontsize=16)
plt.ylabel("Average AUC Gain Across Methods", fontsize=16)
plt.title("Correlation between Bias and Average Gain", fontsize=18)

# Annotate correlation
plt.text(
    0.80, 0.20,
    f"r = {r:.2f}\nR² = {r2:.2f}\n(p = {p:.3f})",
    transform=plt.gca().transAxes,
    fontsize=18,
    verticalalignment="top",
    bbox=dict(facecolor='white', alpha=0.7, edgecolor='gray')
)

plt.xticks(fontsize=25)
plt.yticks(fontsize=25)
plt.legend(title="Feature Type", fontsize=12, title_fontsize=18, )
plt.tight_layout()
plt.savefig(os.path.join(root, "Bias_vs_AverageGain_scatter_colored_gap.svg"))
plt.show()


In [ ]:
## brain Feature importance plots

# ============================================================
# GLOBAL SETTINGS
# ============================================================
import os
import gc
import math
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

from scipy.stats import ttest_rel
from sklearn.metrics import mean_absolute_error
from matplotlib.ticker import FormatStrFormatter

from nilearn import image as nli
from nilearn import plotting
from nilearn.datasets import (
    fetch_surf_fsaverage,
    fetch_atlas_surf_destrieux
)
from nilearn.surface import load_surf_data
from surfplot import Plot

# ---------- SVG + font polish ----------
mpl.rcParams.update({
    "svg.fonttype": "none",
    "font.size": 14
})

# ---------- FIGURE SIZES ----------
# surfplot uses PIXELS
SURFPLOT_SIZE = {
    "surface": (1200, 350),   # cortical 
}

# matplotlib / nilearn use INCHES
MPL_FIGSIZE = {
    "subcortical": (6, 6),
    "connectome": (6, 6),
}

# ============================================================
# PATHS
# ============================================================
root = '/media/hcs-sci-psy-narun/ABCC/fmriresults01/derivatives/ML_Tables/Ethnicity/tsp'
savepath = f'{root}/FI/incr/step100/'
os.makedirs(savepath, exist_ok=True)

modalities  = ['abccSmri']#'abccConnN','abcdRsmri',  'abccCntr','abcdCntr', ,'abccCntr', , 'abccGtfcN', 'abcdCntr''abccConnN', 'abcdRsmri', , 'abccGtfcN', , 'abccSmri''abccSmri''abcdRsmri', 'abccSmri'


def average_weight_change(coefa, coeft):
    """
    Average absolute difference between adapted and typical weight changes.
    """
    return np.mean(np.abs(coefa - coeft))

# ============================================================
# DESTREIUX SURFACE
# ============================================================
def plot_dest(coefs, key, top_percent=100):

    coefs_dest = []
    for coef in coefs:
        coef = np.insert(coef, 41, 0)    # LH medial wall
        coef = np.insert(coef, 116, 0)   # RH medial wall
        coefs_dest.append(coef)

    coefs = np.array(coefs_dest).T
    total_change = np.abs(coefs[:, -1] - coefs[:, 0])

    n_top = max(1, int(len(total_change) * top_percent / 100))
    top_idx = np.argsort(total_change)[-n_top:]

    fsaverage = fetch_surf_fsaverage(mesh='fsaverage5')
    atlas = fetch_atlas_surf_destrieux()

    L_labels = load_surf_data(atlas['map_left'])
    R_labels = load_surf_data(atlas['map_right'])

    L_data = np.zeros_like(L_labels, float)
    R_data = np.zeros_like(R_labels, float)

    for label in np.unique(L_labels):
        if label == 0:
            continue
        idx = label - 1
        if idx in top_idx:
            L_data[L_labels == label] = total_change[idx]

    for label in np.unique(R_labels):
        if label == 0:
            continue
        idx = label - 1
        print('idx',idx, label)
        if (idx + 75) in top_idx:
            R_data[R_labels == label] = total_change[idx + 75]

    p = Plot(
        fsaverage['pial_left'],
        fsaverage['pial_right'],
        size=SURFPLOT_SIZE["surface"],  # <-- PIXELS
        layout='row',
        zoom=1.2
    )

    p.add_layer({'left': L_data, 'right': R_data}, cmap='Reds', cbar=True)
    fig = p.build()

    out = os.path.join(savepath, f"{key}_dest_topchanged.svg")
    fig.savefig(out, format="svg", bbox_inches="tight")
    plt.close(fig)

    atlas = fetch_atlas_surf_destrieux()
    labels = atlas["labels"][1:]  # skip "unknown"

    rows = []
    print(total_change.shape)
    for i, v in enumerate(total_change):
        hemi = "LH" if i < 75 else "RH"
        region = labels[i % 75].decode("utf-8")

        rows.append({
            "feature_index": i,
            "hemisphere": hemi,
            "region": region,
            "delta_delta": v,
            # "abs_delta_delta": abs(v),
        })

    pd.DataFrame(rows).to_csv(savepath + f"{key}_dest_topchanged.csv", index=False)

# ============================================================
# SUBCORTICAL
# ============================================================
def plot_subc(coefs, key, top_percent=100):
    atlas_path = '/media/hcs-sci-psy-narun/Alina/atlases/'
    atlas = nli.load_img(atlas_path + 'subc2.fs.nii.gz')
    labels = atlas.get_fdata()
    coefs = np.array(coefs).T
    total_change = np.abs(coefs[:, -1] - coefs[:, 0])
    n_top = max(1, int(len(total_change) * top_percent / 100))
    top_idx = np.argsort(total_change)[-n_top:]
    region_ids = pd.read_csv(atlas_path + 'atlas_index_num.txt')['nn'].values
    data = np.zeros_like(labels)
    for i in top_idx:
        data[labels == region_ids[i]] = total_change[i]
    img = nli.new_img_like(atlas, data)
    
    # Create figure with white background
    fig = plt.figure(figsize=MPL_FIGSIZE["subcortical"], facecolor='white', edgecolor='white')
    fig.patch.set_facecolor('white')
    
    # Plot with white background (black_bg=False), no default colorbar
    display = plotting.plot_glass_brain(
        img,
        cmap='hot_r',
        colorbar=False,
        symmetric_cbar=False,
        display_mode='lyrz',
        black_bg=False,
        figure=fig, 
        # transparency=0
    )
    
    # Explicitly set axes background to white
    for k, ax in display.axes.items():
        ax.ax.set_facecolor('white')
    
    # Shrink space between subplots
    fig.subplots_adjust(wspace=0.05, hspace=0.05, bottom=0.15)
    
    # Manually add horizontal colorbar at the bottom
    vmin = 0  # Assuming data is non-negative; adjust if needed
    vmax = np.max(data)
    sm = plt.cm.ScalarMappable(cmap='hot_r', norm=plt.Normalize(vmin=vmin, vmax=vmax))
    cbar_ax = fig.add_axes([0.1, 0.05, 0.8, 0.05])  # Adjust position 
    fig.colorbar(sm, cax=cbar_ax, orientation='horizontal')
    
    out = os.path.join(savepath, f"{key}_subc_topchanged.svg")
    display.savefig(out)
    display.close()
    
    rows = []
    for i, v in enumerate(total_change):
        rows.append({
            "feature_index": i,
            "region": region_ids[i],
            "structure_id": labels[i],
            "delta_delta": v,
            # "abs_delta_delta": abs(v),
        })
    pd.DataFrame(rows).to_csv(savepath + f"{key}_subc_topchanged.csv", index=False)
# ============================================================
# GLASSER SURFACE
# ============================================================
def plot_glasser(coefs, key, top_percent=100):

    import hcp_utils as hcp
    from neuromaps.datasets import fetch_fslr

    coefs = np.array(coefs).T
    total_change = np.abs(coefs[:, -1] - coefs[:, 0])

    n_top = max(1, int(len(total_change) * top_percent / 100))
    thresh = np.sort(total_change)[-n_top]

    display_data = total_change.copy()
    display_data[display_data < thresh] = 0

    vertex = hcp.unparcellate(display_data, hcp.mmp)
    L = hcp.left_cortex_data(vertex)
    R = hcp.right_cortex_data(vertex)

    surf = fetch_fslr()
    p = Plot(
        surf['midthickness'][0],
        surf['midthickness'][1],
        size=SURFPLOT_SIZE["surface"],  # <-- PIXELS
        views=['lateral', 'medial'],
        layout='row',
        zoom=1.3
    )


    p.add_layer({'left': L, 'right': R}, cmap='Reds')
    fig = p.build()

    out = os.path.join(savepath, f"{key}_glasser_topchanged.svg")
    fig.savefig(out, format="svg", bbox_inches="tight")
    plt.close(fig)

    glasser_labels = hcp.mmp.labels  # length = 379

    rows = []
    for i, v in enumerate(total_change):
        label = glasser_labels[i+1]
        hemi = "LH" if label.startswith("L_") else "RH"

        rows.append({
            "feature_index": i,
            "hemisphere": hemi,
            "region": label,
            "delta_delta": v,
            # "abs_delta_delta": abs(v),
        })

    pd.DataFrame(rows).to_csv(savepath + f"{key}_glasser_topchanged.csv", index=False)

# ============================================================
# CONNECTOME
# ============================================================
from mpl_toolkits.axes_grid1 import make_axes_locatable
def plot_Gconn(coefs, key, node_coords):
    file_dir = '/media/hcs-sci-psy-narun/ABCC/fmriresults01/derivatives/ML_Tables/nesi_outputs/std_applicability/'

    fold_base = '/media/hcs-sci-psy-narun/ABCC/fmriresults01/derivatives/ML_Tables/Ethnicity/tsp/Fold_0/pls/TRB/'

    original_feature_names = joblib.load(file_dir + 'connmat_headers')
    feature_names = original_feature_names[1:]  # drop intercept

    coef0, coef1 = coefs[0].ravel(), coefs[1].ravel()
    abs_change = np.abs(coef1 - coef0)

    top_k = max(1, int(0.0005 * len(abs_change)))
    top_idx = np.argsort(abs_change)[-top_k:]

    n_rois = node_coords.shape[0]
    adj = np.zeros((n_rois, n_rois))
    triu_i, triu_j = np.triu_indices(n_rois, k=1)

    nodes = set()
    for idx in top_idx:
        i, j = triu_i[idx], triu_j[idx]
        adj[i, j] = adj[j, i] = abs_change[idx]
        nodes.update([i, j])

    nodes = sorted(nodes)
    adj = adj[np.ix_(nodes, nodes)]
    coords = node_coords[nodes]

    # Create figure
    fig = plt.figure(figsize=MPL_FIGSIZE["connectome"])
    ax = fig.add_subplot(111)
    node_color = ["#b7da85" if x < 0 else "#85c8da" for x, y, z in coords]

    display = plotting.plot_connectome(
        adj,
        coords,
        node_size=5,
        node_color= '#677066',
        display_mode = 'lzry',
        edge_cmap='Reds',
        edge_vmin=0,
        edge_vmax=adj.max(),
        colorbar=False,   # Disable default colorbar
        axes=ax
    )

    # ---- Custom bottom colorbar ----
    divider = make_axes_locatable(ax)
    cax = divider.append_axes("bottom", size="5%", pad=0.3)

    sm = plt.cm.ScalarMappable(
        cmap="Reds",
        norm=plt.Normalize(vmin=0, vmax=adj.max())
    )
    sm.set_array([])

    cbar = plt.colorbar(sm, cax=cax, orientation="horizontal")

    # Set only 3 ticks: min, midpoint, max
    vmin = 0
    vmax = adj.max()
    ticks = [vmin, (vmin + vmax) / 2, vmax]

    cbar.set_ticks(ticks)
    cbar.set_ticklabels([f"{t:.5f}" for t in ticks])

    cbar.set_label("Absolute Coefficient Change", fontsize=14)
    cbar.ax.tick_params(labelsize=10)

    plt.show()
    out = os.path.join(savepath, f"{key}_gconn_topchanged.svg")
    display.savefig(out)
    display.close()

    rows = []
    for i, (name, v) in enumerate(zip(feature_names, abs_change)):
        roi1, roi2 = name.split("_&_")
        rows.append({
            "feature_index": i,
            "ROI1": roi1,
            "ROI2": roi2,
            "delta_delta": v,
            # "abs_delta_delta": abs(v),
        })

    pd.DataFrame(rows).to_csv(savepath + f"{key}_gconn_topchanged.csv", index=False)

# ============================================================
# MAIN LOOP
# ============================================================
# ============================================================
# LOAD PLS OUTPUTS
# ============================================================
df_all = pd.read_csv('/media/hcs-sci-psy-narun/ABCC/fmriresults01/derivatives/ML_Tables/Ethnicity/tsp/plots/ALLMODS_TypicalPLS_Step0_GapRanking_rep0_SUBJECTBOOT.csv', index_col=1)

selected_features = list(l1_labels.keys())
df_all = df_all.loc[selected_features]#
df_all =  df_all.sort_values(by='gap')
least_biased = list(df_all.index)[0:10]
most_biased = list(df_all.index)[-10:]
selected_features = least_biased + most_biased
summary_rows = []
for mod in modalities:
    typ = joblib.load(
        f'{root}/Fold_0/pls/TRB/total_{mod}_pls_output_std_All_noDA_incr30f.joblib'
    )
    bw = joblib.load(
        f'{root}/Fold_0/pls/TRB/total_{mod}_pls_output_std_All_BW_incr10f.joblib'
    )

    for key in bw.keys():

        if key not in selected_features:
            continue
        print(key)
        coefs = []
        # Get the last versus first step of AA addition
        coeft = typ[key]['AA_100']['model']['best_coefs'][0] - typ[key]['AA_10']['model']['best_coefs'][0]
        coefa = bw[key]['AA_100']['model']['best_coefs'][0] - bw[key]['AA_10']['model']['best_coefs'][0]
        avg_change = average_weight_change(coefa, coeft)
        summary_rows.append({
            "modality": mod,
            "phenotype": key,
            "avg_abs_weight_change": avg_change,
            "n_features": coefa.shape[0]
        })

        coefs = [coefa, coeft]
        p = coefa.shape[0]
        # plot:
        if p in [148, 167]:
            plot_dest(coefs, key)
        elif p == 379:
            plot_glasser(coefs, key)
        if p == 19:
            plot_subc(coefs, key)
        if p == 71631:
            plot_Gconn(coefs, key, node_coords)
        if key == 'DTI':
            print('DTI')
            coef0, coef1 = coefs[0].ravel(), coefs[1].ravel()
            abs_change = np.abs(coef1 - coef0).reshape(-1, 1)
            pd.DataFrame(abs_change).to_csv(savepath + 'DTI_abs_change.csv')
        # summations
        if p == 9 or p == 4:
            coef0, coef1 = coefs[0].ravel(), coefs[1].ravel()
            abs_change = np.abs(coef1 - coef0).reshape(-1, 1)  # make it (n, 1)
        #     
            sns.heatmap(
                abs_change,
                cmap="RdBu_r",
                center=0,
                # vmin=-vmax,
                # vmax=vmax,
                #yticklabels=yticks,
                # xticklabels=AA_REF,
                cbar=True,
                cbar_kws=dict()#shrink=0.6, pad=0.02
            )
            
            out = os.path.join(savepath, f"{key}_topchanged.svg")
            plt.savefig(out)
            plt.show()

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(savepath + "avg_weight_change_per_phenotype.csv", index=False)
gc.collect()
